![Traffic](traffic.png)

Traffic data fluctuates constantly or is affected by time. Predicting it can be challenging, but this task will help sharpen your time-series skills. With deep learning, you can use abstract patterns in data that can help boost predictability.

Your task is to build a system that can be applied to help you predict traffic volume or the number of vehicles passing at a specific point and time. Determining this can help reduce road congestion, support new designs for roads or intersections, improve safety, and more! Or, you can use to help plan your commute to avoid traffic!

The dataset provided contains the hourly traffic volume on an interstate highway in Minnesota, USA. It also includes weather features and holidays, which often impact traffic volume.

Time to predict some traffic!

### The data:

The dataset is collected and maintained by UCI Machine Learning Repository. The target variable is `traffic_volume`. The dataset contains the following and has already been normalized and saved into training and test sets:

`train_scaled.csv`, `test_scaled.csv`
| Column     | Type       | Description              |
|------------|------------|--------------------------|
|`temp`                   |Numeric            |Average temp in kelvin|
|`rain_1h`                |Numeric            |Amount in mm of rain that occurred in the hour|
|`snow_1h`                |Numeric            |Amount in mm of snow that occurred in the hour|
|`clouds_all`             |Numeric            |Percentage of cloud cover|
|`date_time`              |DateTime           |Hour of the data collected in local CST time|
|`holiday_` (11 columns)  |Categorical        |US National holidays plus regional holiday, Minnesota State Fair|
|`weather_main_` (11 columns)|Categorical     |Short textual description of the current weather|
|`weather_description_` (35 columns)|Categorical|Longer textual description of the current weather|
|`hour_of_day`|Numeric|The hour of the day|
|`day_of_week`|Numeric|The day of the week (0=Monday, Sunday=6)|
|`day_of_month`|Numeric|The day of the month|
|`month`|Numeric|The number of the month|
|`traffic_volume`         |Numeric            |Hourly I-94 ATR 301 reported westbound traffic volume|

In [20]:
# Import the relevant libraries
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

In [21]:
# Read the traffic data from the CSV training and test files
train_scaled_df = pd.read_csv('train_scaled.csv')
test_scaled_df = pd.read_csv('test_scaled.csv')

train_scaled_df.head(20)

,temp,rain_1h,snow_1h,clouds_all,holiday_Christmas Day,holiday_Columbus Day,holiday_Independence Day,holiday_Labor Day,holiday_Martin Luther King Jr Day,holiday_Memorial Day,holiday_New Years Day,holiday_State Fair,holiday_Thanksgiving Day,holiday_Veterans Day,holiday_Washingtons Birthday,weather_main_Clear,weather_main_Clouds,weather_main_Drizzle,weather_main_Fog,weather_main_Haze,weather_main_Mist,weather_main_Rain,weather_main_Smoke,weather_main_Snow,weather_main_Squall,weather_main_Thunderstorm,weather_description_SQUALLS,weather_description_Sky is Clear,weather_description_broken clouds,weather_description_drizzle,weather_description_few clouds,weather_description_fog,weather_description_haze,weather_description_heavy intensity drizzle,weather_description_heavy intensity rain,weather_description_heavy snow,weather_description_light intensity drizzle,weather_description_light intensity shower rain,weather_description_light rain,weather_description_light rain and snow,weather_description_light shower snow,weather_description_light snow,weather_description_mist,weather_description_moderate rain,weather_description_overcast clouds,weather_description_proximity shower rain,weather_description_proximity thunderstorm,weather_description_proximity thunderstorm with drizzle,weather_description_proximity thunderstorm with rain,weather_description_scattered clouds,weather_description_shower drizzle,weather_description_shower snow,weather_description_sky is clear,weather_description_smoke,weather_description_snow,weather_description_thunderstorm,weather_description_thunderstorm with heavy rain,weather_description_thunderstorm with light drizzle,weather_description_thunderstorm with light rain,weather_description_thunderstorm with rain,weather_description_very heavy rain,hour_of_day,day_of_week,day_of_month,month,traffic_volume
0,0.935245,0.0,0.0,0.40,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.391304,0.166667,0.033333,0.818182,0.761676
1,0.938749,0.0,0.0,0.75,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.434783,0.166667,0.033333,0.818182,0.620330
2,0.939463,0.0,0.0,0.90,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.478261,0.166667,0.033333,0.818182,0.654808
3,0.941247,0.0,0.0,0.90,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.521739,0.166667,0.033333,0.818182,0.690385
4,0.944524,0.0,0.0,0.75,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.565217,0.166667,0.033333,0.818182,0.675549
5,0.946405,0.0,0.0,0.01,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.608696,0.166667,0.033333,0.818182,0.711676
6,0.951110,0.0,0.0,0.01,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.652174,0.166667,0.033333,0.818182,0.767033
7,0.953348,0.0,0.0,0.01,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1

In [22]:
# Convert the DataFrame to NumPy arrays
train_scaled = train_scaled_df.to_numpy()
test_scaled = test_scaled_df.to_numpy()

# Start coding here
# Use as many cells as you like!

In [23]:
# The code is taking a long time because the create_seq function appends NumPy array slices
# to Python lists in a loop, which is inefficient for large datasets.
# Additionally, the function returns lists of arrays, which are slower to process and use more memory.
# It is better to pre-allocate NumPy arrays for X and y, and fill them in the loop.

# Here is an optimized version:

def create_seq_optimized(samples, seq_len):
    num_samples = len(samples) - seq_len
    num_features = samples.shape[1] - 1  # Exclude the target column
    X = np.zeros((num_samples, seq_len, num_features), dtype=samples.dtype)
    y = np.zeros(num_samples, dtype=samples.dtype)
    for i in range(num_samples):
        X[i] = samples[i:i+seq_len, :-1]
        y[i] = samples[i+seq_len, -1]
    return X, y

X_train, y_train = create_seq_optimized(train_scaled, 15)
X_test, y_test = create_seq_optimized(test_scaled, 15)

# Now X_train, y_train, X_test, y_test are NumPy arrays and the function is much faster.
# Avoid printing large arrays directly in a notebook cell.
X_train.shape, y_train.shape, X_test.shape, y_test.shape

((34027, 15, 65), (34027,), (6518, 15, 65), (6518,))

In [24]:
# Creating an LSTM model to train on our data
class lstm_model(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm_layer = nn.LSTM(
            input_size = 65,
            hidden_size = 64,
            num_layers = 6,
            batch_first = True,
        )
    
        self.linear_layer = nn.Sequential(
                            nn.ReLU(),
                            nn.Linear(in_features = 64,
                                     out_features = 1)
        )

    def forward(self,x):
        c_0 = torch.zeros(15,x.size(0),64)
        h_0 = torch.zeros(15,x.size(0),64)
        out,_ = self.lstm_layer(x,(h_0,c_0))
        out = self.linear_layer(out[:,-1,:])
        return out

traffic_model = lstm_model()

In [25]:
# Creating datasets and DataLoaders
train_data = TensorDataset(torch.Tensor(X_train),torch.Tensor(y_train))
test_data = TensorDataset(torch.Tensor(X_test),torch.Tensor(y_test))

train_dataloader = DataLoader(dataset = train_data,
                             batch_size = 32
                            )
test_dataloader = DataLoader(dataset = test_data,
                            batch_size = 32)


In [26]:
# Setting up our optimizer and loss function
optimizer = torch.optim.Adam(traffic_model.parameters(),lr = 0.001)
loss_fn = nn.MSELoss()

In [27]:
# Setting a training loop
epochs = 1
traffic_model.train()
final_training_loss = 0
for epoch in range(epochs):
    train_loss = 0
    for X,y in train_dataloader:
        y_pred = traffic_model(X)

        loss = loss_fn(y_pred,y)

        optimizer.zero_grad()
        
        loss.backward()
        
        train_loss += loss
        
        optimizer.step()
        
    final_training_loss += train_loss/len(train_dataloader)

final_training_loss /= epochs

Error: Failed to execute this cell, please try again.

In [28]:
# Calculating eval loss
traffic_model.eval()
with torch.inference_mode():
    test_mse = 0
    for X,y in test_dataloader:
        y_pred = traffic_model(X)
        loss = loss_fn(y_pred,y)
        test_mse += loss
        
    test_mse /= len(test_dataloader)    

In [29]:
print(final_training_loss)
print(test_mse)

0.07561003388942016
tensor(0.0751)
